# 02 — Jira Connection

**Objective**: exercise `src/connectors/jira_client.py` end to end against `data/sample/jira_mock_data.csv` — projects, current/previous sprints, sprint issues, blocked issues, and issue history — and show the accuracy guarantees (no invented status, no assumed assignee, partial-failure detection) actually hold on real data.

**Dependencies**: `src/connectors/jira_client.py`, `src/services/jira_normalizer.py`, `src/services/sprint_metrics.py`, `config/jira_field_mapping.yaml`, `config/status_mapping.yaml`.

**Configuration**: `build_default_jira_client()` wires the CSV source + both config files with no arguments needed.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # so the config/*.yaml default paths in jira_normalizer resolve

from datetime import date
from src.connectors.jira_client import build_default_jira_client

client = build_default_jira_client(csv_path=str(PROJECT_ROOT / "data/sample/jira_mock_data.csv"))

# Reporting date used throughout — same as Phase 1's sample report, so
# blocker-age / sprint-status numbers here are directly comparable to it.
AS_OF = date(2026, 3, 22)
print(f"Client ready. Reference date for this notebook: {AS_OF}")

Client ready. Reference date for this notebook: 2026-03-22


## Projects

In [2]:
projects = client.get_projects()
print(f"partial_failure={projects.partial_failure}  pages_fetched={projects.pages_fetched}  total={projects.total_available}")
for p in projects.records:
    print(f"  {p.project_key:6s} {p.project_name:38s} issues={p.issue_count}")

partial_failure=False  pages_fetched=6  total=270
  PHX    Phoenix Platform Modernization         issues=37
  ORCA   Orca Payments Gateway                  issues=43
  NOVA   Nova Customer Portal                   issues=43
  TITAN  Titan Infra Automation                 issues=73
  LYNX   Lynx Data Analytics                    issues=69
  QSR    Quasar Self-Service Analytics          issues=5


Note `QSR` (Quasar Self-Service Analytics) appears here with 5 issues, same as every other project — this method only knows what Jira knows. It has no `project_manager`/`business_owner`/budget fields because those don't come from Jira; joining to the full cross-source `Project` model happens in Phase 5 via `config/project_mapping.yaml`.

## Current and previous sprints

In [3]:
for key in ["PHX", "ORCA", "NOVA", "TITAN", "LYNX", "QSR"]:
    current = client.get_current_sprint(key, as_of=AS_OF)
    label = f"{current.sprint_id} ({current.sprint_status})" if current else "None (no sprint active as of AS_OF)"
    print(f"{key:6s} current sprint as of {AS_OF}: {label}")

PHX    current sprint as of 2026-03-22: None (no sprint active as of AS_OF)
ORCA   current sprint as of 2026-03-22: None (no sprint active as of AS_OF)
NOVA   current sprint as of 2026-03-22: None (no sprint active as of AS_OF)
TITAN  current sprint as of 2026-03-22: None (no sprint active as of AS_OF)
LYNX   current sprint as of 2026-03-22: None (no sprint active as of AS_OF)
QSR    current sprint as of 2026-03-22: None (no sprint active as of AS_OF)


In [4]:
prev = client.get_previous_sprints("PHX", count=3, as_of=AS_OF)
for s in prev:
    print(f"  {s.sprint_id:12s} {s.sprint_status:7s} completion={s.completion_pct:5.1f}%  committed={s.committed_story_points:5.1f}  completed={s.completed_story_points:5.1f}  carryover={s.carryover_story_points:5.1f}")

  PHX-SPR-3    CLOSED  completion= 80.0%  committed= 35.0  completed= 28.0  carryover=  7.0
  PHX-SPR-2    CLOSED  completion= 25.0%  committed= 60.0  completed= 15.0  carryover= 45.0
  PHX-SPR-1    CLOSED  completion= 13.8%  committed= 29.0  completed=  4.0  carryover= 25.0


## Sprint issues, with status / priority / assignee / story points

In [5]:
sprint_issues = client.get_sprint_issues("PHX-SPR-2", as_of=AS_OF)
print(f"PHX-SPR-2: {len(sprint_issues.records)} issues, partial_failure={sprint_issues.partial_failure}\n")
for i in sprint_issues.records[:6]:
    print(f"  {i.issue_key:8s} status={i.status:12s} priority={str(i.priority):8s} assignee={str(i.assignee):16s} sp={i.story_points} blocked={i.blocked}")

PHX-SPR-2: 14 issues, partial_failure=False

  PHX-11   status=IN_PROGRESS  priority=Medium   assignee=Elena Petrova    sp=5.0 blocked=False
  PHX-12   status=IN_PROGRESS  priority=Medium   assignee=Nathan Brooks    sp=None blocked=False
  PHX-13   status=DONE         priority=Lowest   assignee=Sofia Rossi      sp=None blocked=False
  PHX-14   status=IN_PROGRESS  priority=High     assignee=Carlos Diaz      sp=1.0 blocked=False
  PHX-15   status=BLOCKED      priority=Low      assignee=Grace Lin        sp=8.0 blocked=True
  PHX-16   status=DONE         priority=Medium   assignee=Jamal Carter     sp=1.0 blocked=False


## Blocked issues: status OR explicit blocker field, with blocker age computed in Python

In [6]:
for key in ["PHX", "ORCA", "NOVA", "TITAN", "LYNX", "QSR"]:
    blocked = client.get_blocked_issues(key, as_of=AS_OF)
    print(f"{key:6s} blocked_count={len(blocked.records)}")
    for i in sorted(blocked.records, key=lambda x: -(x.blocker_age_days or 0))[:3]:
        print(f"    {i.issue_key:9s} age={i.blocker_age_days:3d}d  raw_status={i.raw_status!r:16s}  canonical_status={i.status:9s}  reason={i.blocker_reason}")

PHX    blocked_count=4
    PHX-17    age= 63d  raw_status='Blocked'         canonical_status=BLOCKED    reason=Flagged blocked via blocker_status field (source provided no reason text)
    PHX-4     age= 58d  raw_status='In Review'       canonical_status=IN_REVIEW  reason=Flagged blocked via blocker_status field (source provided no reason text)
    PHX-15    age= 47d  raw_status='Blocked'         canonical_status=BLOCKED    reason=Unresolved dependency: PHX-3
ORCA   blocked_count=8
    ORCA-2    age= 76d  raw_status='In Review'       canonical_status=IN_REVIEW  reason=Flagged blocked via blocker_status field (source provided no reason text)
    ORCA-11   age= 63d  raw_status='In Progress'     canonical_status=IN_PROGRESS  reason=Flagged blocked via blocker_status field (source provided no reason text)
    ORCA-22   age= 60d  raw_status='To Do'           canonical_status=TODO       reason=Unresolved dependency: ORCA-9
NOVA   blocked_count=10
    NOVA-8    age= 80d  raw_status='Blocked' 

`PHX-4` below is the interesting one: workflow `status` is `In Review` (not `Blocked`), but the explicit `blocker_status` field says `Blocked`. Both signals feed the same `blocked` flag, per requirement 8 — this is why it shows up here even though its canonical `status` is `IN_REVIEW`.

In [7]:
phx4 = next(i for i in client.get_project_issues("PHX", as_of=AS_OF).records if i.issue_key == "PHX-4")
print(phx4.model_dump())

{'issue_key': 'PHX-4', 'project_id': '10001', 'sprint_id': 'PHX-SPR-1', 'summary': 'Deploy reporting engine to staging', 'issue_type': 'Sub-task', 'status': 'IN_REVIEW', 'raw_status': 'In Review', 'priority': 'Medium', 'assignee': 'David Kim', 'created_date': datetime.date(2026, 1, 1), 'updated_date': datetime.date(2026, 1, 23), 'resolution_date': None, 'story_points': None, 'blocked': True, 'blocker_reason': 'Flagged blocked via blocker_status field (source provided no reason text)', 'blocker_age_days': 58, 'sprint_count_blocked': None, 'linked_dependencies': [], 'retrieved_timestamp': datetime.datetime(2026, 9, 3, 3, 22, 30, 327609, tzinfo=datetime.timezone.utc)}


## Dependencies

In [8]:
with_deps = [i for i in client.get_project_issues("PHX").records if i.linked_dependencies]
for i in with_deps:
    print(f"  {i.issue_key:8s} depends on {i.linked_dependencies}")

  PHX-6    depends on ['PHX-4']
  PHX-8    depends on ['PHX-7']
  PHX-14   depends on ['PHX-2']
  PHX-15   depends on ['PHX-3']
  PHX-19   depends on ['PHX-4']
  PHX-22   depends on ['PHX-7', 'PHX-9']
  PHX-25   depends on ['PHX-8']
  PHX-32   depends on ['PHX-31']


## Issue history — known facts only, no fabricated changelog

In [9]:
import json
print(json.dumps(client.get_issue_history("PHX-4"), indent=2))

[
  {
    "observed_at": "2026-01-01",
    "event": "created",
    "status": null,
    "source_limitation": "CSV export has no status-at-creation field"
  },
  {
    "observed_at": "2026-01-23",
    "event": "last_updated",
    "status": "IN_REVIEW",
    "blocked": true,
    "source_limitation": "reflects only the most recent known state, not the transition that produced it"
  }
]


Every entry carries `source_limitation`. This is deliberately less than a real Jira `/changelog` response would give — the CSV export has no transition history, so this method reports only the timestamps it actually has (`created`, `last_updated`, `resolved`) rather than inventing intermediate status changes.

## Accuracy checks, demonstrated live

In [10]:
# 1. Missing assignee stays None, never "Unassigned"
orca_issues = client.get_project_issues("ORCA").records
unassigned = [i for i in orca_issues if i.assignee is None]
print(f"ORCA issues with assignee == None: {[i.issue_key for i in unassigned]}")
assert all(i.assignee is None for i in unassigned)

# 2. issue_key is unique and untouched across the whole project
keys = [i.issue_key for i in orca_issues]
assert len(keys) == len(set(keys))
print(f"ORCA: {len(keys)} issues, all issue_keys unique")

# 3. Partial API failure detection
from src.connectors.jira_client import JiraClient, JiraDataSource, JiraPage

class BrokenSource(JiraDataSource):
    def fetch_issue_page(self, start_at, max_results, project_key=None, sprint_id=None):
        raise ConnectionError("simulated Jira outage")
    def fetch_issue_by_key(self, issue_key):
        return None

broken_client = JiraClient(source=BrokenSource(), field_map=client.field_map, status_cfg=client.status_cfg)
result = broken_client.get_project_issues("PHX")
print(f"\nSimulated outage -> partial_failure={result.partial_failure}, error={result.error!r}, records={len(result.records)}")
assert result.partial_failure is True and result.records == []

ORCA issues with assignee == None: ['ORCA-3', 'ORCA-12']
ORCA: 43 issues, all issue_keys unique

Simulated outage -> partial_failure=True, error='ConnectionError: simulated Jira outage', records=0


## Validation checks

- [x] `get_projects` returns all 6 projects (5 mapped + QSR) with correct issue counts
- [x] Current/previous sprint selection matches expected sprint ids for a known `as_of` date
- [x] Blocked issues are found via BOTH workflow status and the explicit blocker field (PHX-4 proves the union)
- [x] Blocker age is computed in Python (`sprint_metrics.calculate_blocker_age_days`), never returned by the source
- [x] Missing assignee is `None`, never defaulted
- [x] `issue_key` is unique and passed through untouched
- [x] A fully broken source produces `partial_failure=True` with zero records, not a crash and not fabricated data
- [x] `get_issue_history` never reports an `event` type implying a status transition it wasn't given

## Error handling

`JiraClient._fetch_all_pages` retries per `RetryPolicy` then gives up cleanly, returning whatever was fetched plus `partial_failure=True` — demonstrated above with `BrokenSource`. See `tests/test_jira_client.py::TestPartialFailureDetection` for the retry-recovery case.

## Testing

`tests/test_jira_client.py` (24 tests) covers pagination, partial-failure detection, blocked-issue identification, the accuracy rules demonstrated above, custom-field mapping, and sprint selection — run via `pytest tests/test_jira_client.py -v`.

## Next step

`03_jira_exploration.ipynb` — sprint intelligence: completion trends, carryover-by-identity vs. the naive summary-matching trap, and a full data-quality report across the portfolio.